In [1]:
import pandas as pd

# 1)

In [2]:
listings_clean = pd.read_csv('../data/listings_clean.csv', parse_dates=['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review'])
listings = listings_clean.copy()

In [3]:
stuff = pd.Series([1,2,3,4,5])
print(stuff, stuff.mean())
stuff.loc[len(stuff)] = 1293
print(stuff, stuff.mean())

0    1
1    2
2    3
3    4
4    5
dtype: int64 3.0
0       1
1       2
2       3
3       4
4       5
5    1293
dtype: int64 218.0


In [4]:
print(listings.number_of_reviews.mean().round(1))
print(listings.number_of_reviews.median())

35.8
9.0


Depends... the mean says more about what value is a normal value, but you can expect more values to be around the median, so if that's all you care about, then that's the value you are interested in.

# 2)

In [5]:
listings.number_of_reviews.describe().round(1)

count    4574.0
mean       35.8
std        78.4
min         0.0
25%         2.0
50%         9.0
75%        29.0
max      1293.0
Name: number_of_reviews, dtype: float64

In [6]:
listings.number_of_reviews.quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.99]).round(0)

0.10      0.0
0.25      2.0
0.50      9.0
0.75     29.0
0.90     94.0
0.99    400.0
Name: number_of_reviews, dtype: float64

In [7]:
q1 = listings.number_of_reviews.quantile(0.25)
q3 = listings.number_of_reviews.quantile(0.75)
iqr = q3 - q1
print(listings.number_of_reviews.describe().round(0))
print(round(q1, 1), round(q3, 1), round(iqr, 1))

count    4574.0
mean       36.0
std        78.0
min         0.0
25%         2.0
50%         9.0
75%        29.0
max      1293.0
Name: number_of_reviews, dtype: float64
2.0 29.0 27.0


iqr is how wide of a range the middle 50% of values are spread (29 - 2)
27 is small when compared with max value and mean value, but large when compared to the median value.
10% is 0, meaning at least 10% of listings have 0 reviews.

# 3)

In [8]:
upper_fence = q3 + 1.5 * iqr
lower_fence = q1 - 1.5 * iqr
print(round(lower_fence, 1), round(upper_fence, 1))

-38.5 69.5


number of reviews cannot be lower than 0, so we can not have values below lower_fence

In [9]:
outliers = listings[listings.number_of_reviews > upper_fence]
print(len(outliers))

582


In [10]:
outliers.groupby('room_type').agg(
    num=('number_of_reviews', 'count'),
    median_price=('price', 'median')
)

,num,median_price
room_type,,
Entire home/apt,446,1889.5
Hotel room,1,NaN
Private room,121,891.0
Shared room,14,297.0


In [11]:
wihtout_outliers = listings[listings.number_of_reviews <= upper_fence]
print(len(wihtout_outliers))

3992


In [12]:
wihtout_outliers.groupby('room_type').agg(
    num=('number_of_reviews', 'count'),
    median_price=('price', 'median')
)

,num,median_price
room_type,,
Entire home/apt,3314,2282.835
Hotel room,2,859.000
Private room,666,802.250
Shared room,10,304.000


no significant difference in median_price with or without the "number_of_review soutliers"

# 4)

In [13]:
listings['reviews_outlier'] = listings.number_of_reviews > upper_fence

In [14]:
print(listings.reviews_outlier.sum())

582


In [15]:
compare = pd.DataFrame({
    'all': listings.number_of_reviews.agg(['count', 'mean', 'median', 'std']),
    'without_outlers': listings.loc[~listings['reviews_outlier'], 'number_of_reviews'].agg(['count', 'mean', 'median', 'std'])
}).round(0)
compare

,all,without_outlers
count,4574.0,3992.0
mean,36.0,13.0
median,9.0,7.0
std,78.0,16.0


In [16]:
print(round(3992.0/4574.0, 2), round(13.0/36.0, 2), round(7.0/9.0, 2), round(16.0/78.0, 2))

0.87 0.36 0.78 0.21


count and median changed the least, mean and std changed the most (as expected)

# 5)

In [17]:
listings.groupby('room_type')['number_of_reviews'].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
room_type,,,,,,,,
Entire home/apt,3760.0,34.0,76.0,0.0,2.0,8.0,27.0,1293.0
Hotel room,3.0,61.0,16.0,49.0,52.0,56.0,68.0,79.0
Private room,787.0,41.0,84.0,0.0,2.0,12.0,38.0,867.0
Shared room,24.0,114.0,122.0,0.0,28.0,82.0,146.0,510.0


Can't say much about Hotel rooms since count is only 3

# 6)

In [18]:
listings['days_since_review'] = listings.last_scraped - listings.last_review
columns = ['price', 'accommodates', 'bedrooms', 'availability_365', 'review_scores_rating', 'minimum_nights', 'days_since_review']

In [51]:
for col in columns:
    print(f'{col:<21} {listings[col].corr(listings.number_of_reviews).round(2):>5}')


price                 -0.09
accommodates          -0.05
bedrooms              -0.14
availability_365       0.18
review_scores_rating  -0.04
minimum_nights        -0.07
days_since_review      0.18


number_of_reviews has strongest correlation with availability_365 and days_since_review

In [54]:
print(f'without outliers: {wihtout_outliers.availability_365.corr(wihtout_outliers.number_of_reviews).round(2):>5}')
print(f'with outliers: {listings.availability_365.corr(listings.number_of_reviews).round(2):>5}')

without outliers:   0.0
with outliers:  0.18


small correlation with outliers, no correlation without outliers, not much to say either way, numbers are close to 0 either way.

In [55]:
listings.to_csv('../data/listings_clean_outlier.csv', index=False)